# Construyendo Aplicaciones de IA con Gradio y Gemini
 
En los notebooks anteriores interactuamos con LLMs directamente desde Python.
En este notebook construiremos interfaces web interactivas usando **Gradio**,
conectadas a **Gemini 2.5 Flash**.
 
Gradio es una biblioteca de Python que permite crear interfaces web para modelos
de IA con muy poco código. Es ampliamente usada para demos, prototipos y herramientas
internas.
 
![Alt text](figs/01/gradio_gemini_architecture.svg)  
## Prerrequisitos
 
Asegúrate de tener tu `GEMINI_API_KEY` en el archivo `.env`.
Si no la tienes, revisa las instrucciones en el notebook 03.

In [ ]:
# %%bash
# pip install gradio google-genai python-dotenv

## 1. Configuración del Entorno

In [1]:
import os
import gradio as gr
from google import genai
from google.genai import types
from dotenv import load_dotenv

In [ ]:
# Cargamos las variables de entorno desde el archivo .env
load_dotenv()

In [ ]:
# Verificamos que la key esté disponible
if os.getenv("GEMINI_API_KEY"):
    print("Gemini API Key cargada correctamente")
else:
    print("ERROR: GEMINI_API_KEY no encontrada. Verifica tu archivo .env")

In [4]:
# Inicializamos el cliente de Gemini
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [5]:
# Modelo que usaremos en todo el notebook
MODELO = "gemini-2.5-flash-lite"

## 2. Tu Primera Interfaz con Gradio
 
Antes de conectar el LLM, entendamos cómo funciona Gradio con un ejemplo simple.
 
`gr.Interface` toma una función de Python y la convierte en una interfaz web:
- `fn`: la función a ejecutar cuando el usuario hace click en Submit
- `inputs`: tipo de entrada (textbox, imagen, audio, etc.)
- `outputs`: tipo de salida (texto, imagen, etc.)
"""

In [6]:
# Función simple que devuelve lo que el usuario escribe
def echo(message):
    return f"Dijiste: {message}"

In [7]:
# Creamos la interfaz — Gradio levanta un servidor web local automáticamente
# Al ejecutar esta celda verás un link para abrir la interfaz en el navegador
demo = gr.Interface(
    fn=echo,
    inputs="textbox",
    outputs="textbox",
    title="Echo Bot",
    description="Una interfaz simple que repite lo que escribes",
    flagging_mode="never"  # desactiva el botón de reportar respuestas
)

In [ ]:
demo.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [ ]:
demo.close()

## 3. Chatbot Básico con Gemini
 
Ahora conectamos Gradio con Gemini. La función `get_response` llama a la API
y devuelve el texto — Gradio se encarga de mostrarlo en la interfaz.

In [11]:
def get_response(message: str) -> str:
    """Obtiene una respuesta de Gemini para el mensaje del usuario."""
    response = client.models.generate_content(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un asistente util.",
        ),
        contents=message
    )
    return response.text

In [12]:
# Creamos la interfaz del chatbot básico
# Es exactamente igual al echo bot, solo cambia la función
chatbot = gr.Interface(
    fn=get_response,
    inputs=gr.Textbox(placeholder="Escribe tu pregunta aqui..."),
    outputs="text",
    title="Chatbot con Gemini",
    description="Chatea con un asistente de IA powered by Gemini 2.5 Flash",
    flagging_mode="never"
)

In [ ]:
# Lanzamos el chatbot
chatbot.launch()

In [ ]:
chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [ ]:
chatbot.close()

## 4. Chatbot con Streaming
 
El streaming muestra la respuesta a medida que se genera, token por token.
En Gradio esto se logra con `yield` en lugar de `return` — la función se convierte
en un generador que devuelve la respuesta parcial en cada paso.

In [17]:
def stream_response(message: str):
    """Genera una respuesta de Gemini en modo streaming."""
 
    respuesta = ""
 
    # generate_content_stream devuelve chunks a medida que el modelo genera
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un asistente util.",
        ),
        contents=message
    ):
        if chunk.text:
            respuesta += chunk.text
            yield respuesta  # yield devuelve el acumulado parcial a Gradio

In [18]:
# Gradio detecta automáticamente que la función es un generador (usa yield)
# y activa el modo streaming en la interfaz
streaming_chatbot = gr.Interface(
    fn=stream_response,
    inputs="textbox",
    outputs="text",
    title="Chatbot con Streaming",
    description="Las respuestas aparecen en tiempo real a medida que se generan",
    flagging_mode="never"
)


In [ ]:
streaming_chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [ ]:
streaming_chatbot.close()

## 5. Chatbot con Memoria
 
`gr.ChatInterface` es un componente de alto nivel de Gradio diseñado específicamente
para chatbots con historial de conversación. Maneja automáticamente la visualización
del historial — nosotros solo nos encargamos de la lógica del chat.
 
La función recibe:
- `message`: el mensaje actual del usuario
- `history`: lista de intercambios previos en formato `[user_msg, assistant_msg]`

In [31]:
def get_text(content):
    """Extrae el texto sin importar si llega como string o lista."""
    if isinstance(content, str):
        return content
    elif isinstance(content, list):
        return content[0].get("text", "") if content else ""
    return str(content)

def chat_con_memoria(message: str, history: list):
    
    contenido = []
    
    for entry in history:
        if isinstance(entry, dict):
            role = "model" if entry["role"] == "assistant" else "user"
            contenido.append(
                types.Content(role=role, parts=[types.Part(text=get_text(entry["content"]))])
            )
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            user_msg, assistant_msg = entry
            contenido.append(
                types.Content(role="user", parts=[types.Part(text=get_text(user_msg))])
            )
            if assistant_msg:
                contenido.append(
                    types.Content(role="model", parts=[types.Part(text=get_text(assistant_msg))])
                )

    contenido.append(
        types.Content(role="user", parts=[types.Part(text=message)])
    )

    respuesta = ""
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un asistente util.",
        ),
        contents=contenido
    ):
        if chunk.text:
            respuesta += chunk.text
            yield respuesta

In [32]:
# gr.ChatInterface se encarga de toda la UI del chat:
# burbuja de mensajes, historial visual, campo de entrada, etc.
memory_chatbot = gr.ChatInterface(
    fn=chat_con_memoria,
    title="Chatbot con Memoria",
    description="El asistente recuerda el contexto de toda la conversacion.",
    examples=[
        "Cuentame sobre el aprendizaje automatico",
        "Como funcionan las redes neuronales?",
        "Cual es la diferencia entre IA y ML?"
    ],
    flagging_mode="never"
)

In [ ]:
memory_chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [ ]:
memory_chatbot.close()

## 6. Generador de Menús de Restaurante
 
Veamos un ejemplo de aplicación con múltiples entradas. Gradio puede manejar
formularios complejos — aquí el usuario proporciona tres campos y obtiene
un menú generado por IA en formato Markdown.

In [35]:
def generate_menu(nombre_restaurante: str, tipo_cocina: str, requisitos_especiales: str = "Ninguno") -> str:
    """Genera un menu de restaurante usando Gemini."""
 
    prompt = f"""
    Crea un menu de restaurante para "{nombre_restaurante}", un restaurante de cocina {tipo_cocina}.
    Requisitos especiales: {requisitos_especiales}
 
    Incluye:
    - 3 entradas
    - 4 platos principales
    - 2 postres
 
    Para cada item incluye nombre, descripcion breve y precio en pesos colombianos.
 
    Formatea la respuesta en markdown con encabezados, secciones y estilos apropiados.
    Agrega una breve introduccion del restaurante al inicio.
    """
 
    response = client.models.generate_content(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un consultor experto en restaurantes que crea menus hermosos y bien formateados.",
        ),
        contents=prompt
    )
 
    return response.text

In [36]:
# Interfaz con múltiples inputs y output en Markdown
menu_generator = gr.Interface(
    fn=generate_menu,
    inputs=[
        gr.Textbox(label="Nombre del Restaurante"),
        gr.Textbox(label="Tipo de Cocina (ej. italiana, japonesa, colombiana)"),
        gr.Textbox(
            label="Requisitos Especiales (Opcional)",
            placeholder="ej. opciones vegetarianas, sin gluten"
        )
    ],
    outputs=gr.Markdown(label="Menu Generado"),
    title="Generador de Menus con IA",
    description="Crea un menu profesional para tu restaurante en segundos",
    flagging_mode="never"
)

In [ ]:
menu_generator.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [ ]:
menu_generator.close()

## 7. Personalizando el System Prompt
 
Una extensión útil: permitir al usuario definir el comportamiento del asistente
cambiando el system prompt desde la interfaz.

In [39]:
def chat_personalizable(message: str, history: list, system_prompt: str):
    """Chat donde el usuario puede personalizar el system prompt."""
 
    # Si no se provee system prompt, usamos uno por defecto
    if not system_prompt:
        system_prompt = "Eres un asistente util."
 
    # Construimos el historial en formato Gemini
    contenido = []
    for user_msg, assistant_msg in history:
        contenido.append(
            types.Content(role="user", parts=[types.Part(text=user_msg)])
        )
        if assistant_msg:
            contenido.append(
                types.Content(role="model", parts=[types.Part(text=assistant_msg)])
            )
 
    contenido.append(
        types.Content(role="user", parts=[types.Part(text=message)])
    )
 
    # Streaming con el system prompt personalizado
    respuesta = ""
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
        ),
        contents=contenido
    ):
        if chunk.text:
            respuesta += chunk.text
            yield respuesta

In [41]:
# Usamos gr.ChatInterface con additional_inputs para agregar el campo de system prompt
custom_chatbot = gr.ChatInterface(
    fn=chat_personalizable,
    title="Chatbot Personalizable",
    description="Define como quieres que se comporte el asistente.",
    additional_inputs=[
        gr.Textbox(
            label="System Prompt",
            placeholder="ej. Eres un experto en Python que responde solo con codigo",
            value="Eres un asistente util y conciso."
        )
    ],
    examples=[
        ["Explica que es una red neuronal", "Eres un profesor universitario."],
        ["Escribe un haiku sobre la programacion", "Eres un poeta."],
    ],
    flagging_mode="never"
)

In [ ]:
custom_chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [ ]:
custom_chatbot.close()